# Dataset 2 - per-p embeddings (graphsage_v1_64)

Re-runs the **exact same** GraphSAGE extraction used in notebook 08 (`graphsage_v1_64`) once for **each p**. Same network, same features (`assets`, `liabilities`, `buffer`), same config - only the merged target changes at the end. Outputs one parquet per p next to the p0 baseline `graphsage_v1_64_dataset2_dataset.parquet`.

In [1]:
import sys
from pathlib import Path

def find_project_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for path in [start, *start.parents]:
        if (path / 'src').exists() and (path / 'requirements.txt').exists():
            return path
    raise FileNotFoundError('Project root not found.')

PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd

from src.models.embeddings import GNNConfig, extract_embeddings

EMB_ROOT = PROJECT_ROOT / 'src' / 'data' / 'embeddings'
DATASET2_PATH = PROJECT_ROOT / 'src' / 'datasets' / 'dataset_2'
TARGETS_DIR = DATASET2_PATH / 'targets'

def emb_path(name):
    return EMB_ROOT / 'dataset_2' / 'feature_based' / name

def load_dataset2_nodes(dataset_path):
    nodes = pd.read_csv(dataset_path / 'nodes.csv').reset_index(drop=True)
    nodes['index'] = nodes.index
    nodes['Equity'] = nodes['buffer']; nodes['Assets'] = nodes['assets']
    return nodes

def load_dataset2_edges(dataset_path, bank_to_idx):
    matrix = pd.read_excel(dataset_path / 'network.xlsx', index_col=0)
    el = matrix.stack().reset_index(); el.columns = ['source_bank', 'target_bank', 'Weights']
    el = el[el['Weights'] != 0].copy()
    el['Sourceid'] = el['source_bank'].map(bank_to_idx); el['Targetid'] = el['target_bank'].map(bank_to_idx)
    el = el.dropna(subset=['Sourceid', 'Targetid'])
    el['Sourceid'] = el['Sourceid'].astype(int); el['Targetid'] = el['Targetid'].astype(int)
    return el[['Sourceid', 'Targetid', 'Weights']].reset_index(drop=True)

nodes = load_dataset2_nodes(DATASET2_PATH)
edges = load_dataset2_edges(DATASET2_PATH, dict(zip(nodes['bank'], nodes['index'])))
FEATURE_COLS = ['assets', 'liabilities', 'buffer']
TARGET_COL = 'log_systemic_risk_label'
P_LIST = ['p5', 'p10', 'p15', 'p20', 'p25', 'p30', 'p35', 'p40']

# Same config as the selected graphsage_v1_64 embedding (see notebooks 03/08).
cfg_graphsage_v1_64 = GNNConfig(hidden_dims=(256, 64), dropout=0.3, lr=0.01,
                                epochs=100, aggregation='mean', device='cpu')

In [2]:
for p in P_LIST:
    target_p = pd.read_csv(TARGETS_DIR / f'target_{p}.csv')
    emb, _ = extract_embeddings(edges, nodes, cfg_graphsage_v1_64, feature_cols=FEATURE_COLS)
    emb_cols = [c for c in emb.columns if c.startswith('emb_')]
    merged = (emb[['bank_id'] + emb_cols]
              .merge(target_p, on='bank_id', how='inner')[['bank_id'] + emb_cols + [TARGET_COL]])
    out = emb_path(f'graphsage_v1_64_dataset2_{p}_dataset.parquet')
    merged.to_parquet(out, index=False)
    print(f'{p:4s}  shape={merged.shape}  emb_cols={len(emb_cols)}  -> {out.name}')

p5    shape=(1444, 66)  emb_cols=64  -> graphsage_v1_64_dataset2_p5_dataset.parquet


p10   shape=(1444, 66)  emb_cols=64  -> graphsage_v1_64_dataset2_p10_dataset.parquet


p15   shape=(1444, 66)  emb_cols=64  -> graphsage_v1_64_dataset2_p15_dataset.parquet


p20   shape=(1444, 66)  emb_cols=64  -> graphsage_v1_64_dataset2_p20_dataset.parquet


p25   shape=(1444, 66)  emb_cols=64  -> graphsage_v1_64_dataset2_p25_dataset.parquet


p30   shape=(1444, 66)  emb_cols=64  -> graphsage_v1_64_dataset2_p30_dataset.parquet


p35   shape=(1444, 66)  emb_cols=64  -> graphsage_v1_64_dataset2_p35_dataset.parquet


p40   shape=(1444, 66)  emb_cols=64  -> graphsage_v1_64_dataset2_p40_dataset.parquet


In [3]:
files = sorted(emb_path('').glob('graphsage_v1_64_dataset2_p*_dataset.parquet'))
for f in files:
    print(f'{f.name:50s}  shape={pd.read_parquet(f).shape}')

graphsage_v1_64_dataset2_p10_dataset.parquet        shape=(1444, 66)
graphsage_v1_64_dataset2_p15_dataset.parquet        shape=(1444, 66)
graphsage_v1_64_dataset2_p20_dataset.parquet        shape=(1444, 66)
graphsage_v1_64_dataset2_p25_dataset.parquet        shape=(1444, 66)
graphsage_v1_64_dataset2_p30_dataset.parquet        shape=(1444, 66)
graphsage_v1_64_dataset2_p35_dataset.parquet        shape=(1444, 66)
graphsage_v1_64_dataset2_p40_dataset.parquet        shape=(1444, 66)
graphsage_v1_64_dataset2_p5_dataset.parquet         shape=(1444, 66)
